# Step 2a · Review a finished SAC run (no retraining)

Use this when training finished but a later cell failed.

1. Settings: **Accelerator → None (CPU)**, **Internet → On**.
2. Right panel: **Add Input → Your Work → Notebooks →** your `step2_train_sac` notebook (pick the version that trained). Its output files appear under `/kaggle/input/`.
3. Run all cells (interactive is fine: this takes a few minutes).

In [ ]:
# 1. Fresh clone of the repo
!rm -rf robot-world-model
!git clone -q https://github.com/Swayam-27/robot-world-model.git
%cd robot-world-model
!git log --oneline -1

In [ ]:
# 2. Install MuJoCo + Stable-Baselines3 (Kaggle already has PyTorch, pandas, matplotlib)
!pip install -q "gymnasium[mujoco]" stable-baselines3
!python -c "import gymnasium, mujoco, stable_baselines3, torch; print('gymnasium', gymnasium.__version__, '| mujoco', mujoco.__version__, '| sb3', stable_baselines3.__version__, '| torch', torch.__version__)"

In [ ]:
# 3. Find the trained run in the attached input and copy it into outputs/ (inputs are read-only)
import glob, shutil
from pathlib import Path
hits = glob.glob("/kaggle/input/**/sac_pusher/sac_final.zip", recursive=True)
assert hits, "no sac_final.zip found: is the training notebook attached as input? If not saved at all, retrain."
src = Path(hits[0]).parent
shutil.copytree(src, "outputs/sac_pusher", dirs_exist_ok=True)
print("copied from", src)
!ls outputs/sac_pusher outputs/sac_pusher/ckpts

In [ ]:
# 4. Learning curve (the old run named its log monitor.monitor.csv)
mon = sorted(glob.glob("outputs/sac_pusher/*monitor.csv"))[0]
!python -m policies.plot_curve --monitor {mon}
from IPython.display import Image, display
display(Image(filename=str(Path(mon).parent / "learning_curve.png")))

In [ ]:
# 5. Watch early, middle and final checkpoints on a TRAINING seed (eval seeds are only for scoring)
# Each run tries EGL; if it crashes, it reruns with OSMesa (software rendering). The output line says which one worked.
!apt-get -qq install -y libosmesa6 > /dev/null
!for c in ckpts/sac_20000_steps ckpts/sac_100000_steps ckpts/sac_200000_steps sac_final; do python -m policies.watch_policy --ckpt outputs/sac_pusher/$c.zip 2>/dev/null || MUJOCO_GL=osmesa PYOPENGL_PLATFORM=osmesa python -m policies.watch_policy --ckpt outputs/sac_pusher/$c.zip; done
import glob
from IPython.display import Image, display
for g in sorted(glob.glob("outputs/sac_pusher/**/*.gif", recursive=True)):
    print(g); display(Image(filename=g))

In [ ]:
# 6. Zip to /kaggle/working (outside the repo folder, so re-cloning can't delete it). Download from the Output tab.
!cd outputs && zip -qr /kaggle/working/sac_pusher.zip sac_pusher
!ls -lh /kaggle/working/sac_pusher.zip